In [ ]:
from google.colab import drive

drive.mount('/content/drive')


# Geometry-V1 all-layer direction selection

Run once from top to bottom. This handoff only invokes the bounded artifact-selection runner and retains a zero science denominator.


In [ ]:
import json
import os
import pathlib
import subprocess
import sys
import time

DIRECTION_RUNNER_EXACT = '4be67408ac353bd7adfa7c3662af55d579b888de'
SOURCE_D0_EXACT = '4732211beefbeface95cb842c117b9719e362f1a'
SOURCE_RUN_ID = 'geometry-v1-qk-d0-4732211beefb'
SOURCE_PROTOCOL = 'geometry-v1-qk-d0-all-layer-discovery-v1'
SOURCE_ROOT = pathlib.Path('/content/drive/MyDrive/CEG-WM/Geometry-V1/D0/Geometry-V1-QK-D0-4732211beefb-20260827T064555Z')
OUTPUT_ROOT = pathlib.Path('/content/drive/MyDrive/CEG-WM/Geometry-V1/DIRECTION_ALL_LAYER')
RUNNER_PATH = 'experiments/run_geometry_v1_qk_direction_all_layer_selection_operational.py'
SUCCESS_PREFIX = 'CEGWM_GEOMETRY_V1_DIRECTION_ALL_LAYER '
FAILURE_PREFIX = 'CEGWM_GEOMETRY_V1_DIRECTION_ALL_LAYER_FAILURE '
MAX_CONTROL_BYTES = 1024
HANDOFF_FAILED = False
RUNNER_ATTEMPTED = False

def fail_closed(stage):
    global HANDOFF_FAILED
    if not HANDOFF_FAILED:
        HANDOFF_FAILED = True
        print('CEGWM_GEOMETRY_V1_DIRECTION_ALL_LAYER_HANDOFF_FAILURE ' + json.dumps({'stage': stage, 'error_class': 'handoff_error'}, sort_keys=True, separators=(',', ':')))


In [ ]:
if not HANDOFF_FAILED:
    try:
        if not SOURCE_ROOT.is_dir(): raise RuntimeError()
        repo = pathlib.Path('/content/Geometry-V1-Direction-All-Layer')
        if repo.exists(): raise RuntimeError()
        subprocess.run(['git', 'clone', '--no-checkout', 'https://github.com/RICHAAARC/CEG-WM.git', str(repo)], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        subprocess.run(['git', 'checkout', '--detach', DIRECTION_RUNNER_EXACT], cwd=repo, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        execution_commit = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=repo, check=True, capture_output=True, text=True).stdout.strip()
        checkout_clean = not subprocess.run(['git', 'status', '--porcelain'], cwd=repo, check=True, capture_output=True, text=True).stdout.strip()
        if execution_commit != DIRECTION_RUNNER_EXACT or not checkout_clean: raise RuntimeError()
        runner_path = repo / RUNNER_PATH
        if not runner_path.is_file(): raise RuntimeError()
        source_d0_artifact_identity = {'run_id': SOURCE_RUN_ID, 'execution_exact': SOURCE_D0_EXACT, 'protocol': SOURCE_PROTOCOL}
        runner_execution_identity = {'commit': execution_commit, 'clean': checkout_clean}
    except BaseException:
        fail_closed('checkout')


In [ ]:
if not HANDOFF_FAILED:
    control_read = control_write = None
    try:
        if RUNNER_ATTEMPTED: raise RuntimeError()
        RUNNER_ATTEMPTED = True
        OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
        run_dir = OUTPUT_ROOT / ('Geometry-V1-Direction-All-Layer-' + execution_commit[:12] + '-' + time.strftime('%Y%m%dT%H%M%SZ', time.gmtime()))
        if run_dir.exists(): raise RuntimeError()
        control_read, control_write = os.pipe()
        command = [sys.executable, str(runner_path), '--repo-root', str(repo), '--expected-exact', execution_commit, '--source-root', str(SOURCE_ROOT), '--output-root', str(run_dir), '--control-fd', str(control_write)]
        process = subprocess.Popen(command, cwd=repo, pass_fds=(control_write,), stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        os.close(control_write); control_write = None
        runner_rc = process.wait(timeout=7200)
        control_line = os.read(control_read, MAX_CONTROL_BYTES + 1)
        if len(control_line) > MAX_CONTROL_BYTES: raise RuntimeError()
        if control_line.startswith(SUCCESS_PREFIX.encode('ascii')):
            control = json.loads(control_line[len(SUCCESS_PREFIX):])
        elif control_line.startswith(FAILURE_PREFIX.encode('ascii')):
            control = json.loads(control_line[len(FAILURE_PREFIX):])
        else:
            control = {'status': 'unavailable'}
        terminal = {'source_d0_artifact_identity': source_d0_artifact_identity, 'runner_execution_identity': runner_execution_identity, 'status': control.get('selection_status', control.get('status')), 'selected_layer_paths': control.get('selected_layer_paths', []), 'science_denominator': 0}
        print('CEGWM_GEOMETRY_V1_DIRECTION_ALL_LAYER_TERMINAL ' + json.dumps(terminal, sort_keys=True, separators=(',', ':')))
        if runner_rc != 0 or control.get('status') != 'success': raise RuntimeError()
    except BaseException:
        fail_closed('runner')
    finally:
        if control_write is not None: os.close(control_write)
        if control_read is not None: os.close(control_read)
